# Approval Prediction Training (Multiclass)

This notebook trains a multiclass model with labels: `skill_gap`, `stable`, `reward_milestone`, and exports it to ONNX.

Expected CSV columns (from your SQL query):
- `user_email`, `window_type`, `window_start`, `window_end`
- `leave_total`, `leave_approved`, `leave_rejected`
- `doc_total`, `doc_approved`, `doc_rejected`
- `total_requests`, `total_approved`, `total_rejected`, `overall_approval_rate`


In [ ]:
# Install deps (run once)
!pip -q install pandas numpy scikit-learn skl2onnx onnxruntime

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier

from skl2onnx import to_onnx

## Load CSV

In [ ]:
csv_path = "/content/approvals_dataset.csv"  # upload your CSV to Colab files
df = pd.read_csv(csv_path)

# Drop rows with missing user email
df = df[df["user_email"].notna()].copy()

# Ensure numeric columns are numeric
numeric_cols = [
    "leave_total", "leave_approved", "leave_rejected",
    "doc_total", "doc_approved", "doc_rejected",
    "total_requests", "total_approved", "total_rejected",
    "overall_approval_rate"
]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# Optional: derive reject rate for extra signal
df["overall_reject_rate"] = np.where(
    df["total_requests"] > 0,
    df["total_rejected"] / df["total_requests"],
    0.0
)

df.head()

## Create Labels (Multiclass)

Edit thresholds here if needed.

In [ ]:
MIN_REQUESTS = 3
REJECT_RATE_HIGH = 0.50
REJECT_RATE_LOW = 0.10
APPROVED_HIGH = 10

def label_row(row):
    total = row["total_requests"]
    reject_rate = row["overall_reject_rate"]
    approved = row["total_approved"]
    if total < MIN_REQUESTS:
        return "stable"
    if reject_rate >= REJECT_RATE_HIGH:
        return "skill_gap"
    if reject_rate <= REJECT_RATE_LOW and approved >= APPROVED_HIGH:
        return "reward_milestone"
    return "stable"

df["target"] = df.apply(label_row, axis=1)
df["target"].value_counts()

## Train Model

In [ ]:
feature_cols = [
    "window_type",
    "leave_total", "leave_approved", "leave_rejected",
    "doc_total", "doc_approved", "doc_rejected",
    "total_requests", "total_approved", "total_rejected",
    "overall_approval_rate", "overall_reject_rate"
]

X = df[feature_cols].copy()
y = df["target"].copy()

categorical_features = ["window_type"]
numeric_features = [c for c in feature_cols if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

model = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.1, random_state=42)

clf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)
preds = clf.predict(X_test)
print(classification_report(y_test, preds))
print(confusion_matrix(y_test, preds))

## Export to ONNX

In [ ]:
# Build ONNX model
onnx_model = to_onnx(clf, X[:1], target_opset=15)

with open("approval_prediction.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Saved approval_prediction.onnx")